# Combining Flight and Weather Datasets
Combining the csv files "weather.csv" and "flights.csv" whcih have already been cleaned, on location and date

## 1. Imports & Setup

In [4]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

## 2. Examining Flights Data

In [6]:
flights_df = pd.read_csv("flights.csv")
print(flights_df.info())
print(flights_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3966862 entries, 0 to 3966861
Data columns (total 16 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   Carrier Code                      object 
 1   date                              object 
 2   destination_code                  object 
 3   Scheduled elapsed time (Minutes)  float64
 4   Actual elapsed time (Minutes)     float64
 5   Departure delay (Minutes)         float64
 6   Taxi-Out time (Minutes)           float64
 7   origin_code                       object 
 8   month                             float64
 9   day_of_week                       float64
 10  scheduled_hour                    float64
 11  weather_delayed                   int64  
 12  origin_lat                        float64
 13  origin_lon                        float64
 14  dest_lat                          float64
 15  dest_lon                          float64
dtypes: float64(11), int64(1), object(4)


## 3. Examining Weather Data

In [8]:
weather_df = pd.read_csv("weather.csv")
weather_cols = ["DATE", "LATITUDE", "LONGITUDE", "TEMP", "DEWP", "VISIB", "WDSP", "MXSPD", "GUST", "PRCP", "SNDP", "SLP", "ELEVATION", "Fog", "Rain", "Snow", "Hail", "Thunder", "Tornado"]
weather_df = weather_df[weather_cols].copy()
weather_df["DATE"] = pd.to_datetime(weather_df["DATE"])
print(weather_df.info())
print(weather_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 755327 entries, 0 to 755326
Data columns (total 19 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   DATE       755327 non-null  datetime64[ns]
 1   LATITUDE   755327 non-null  float64       
 2   LONGITUDE  755327 non-null  float64       
 3   TEMP       755302 non-null  float64       
 4   DEWP       752789 non-null  float64       
 5   VISIB      744484 non-null  float64       
 6   WDSP       751208 non-null  float64       
 7   MXSPD      749569 non-null  float64       
 8   GUST       492937 non-null  float64       
 9   PRCP       755327 non-null  float64       
 10  SNDP       7767 non-null    float64       
 11  SLP        447235 non-null  float64       
 12  ELEVATION  755327 non-null  float64       
 13  Fog        755327 non-null  int64         
 14  Rain       755327 non-null  int64         
 15  Snow       755327 non-null  int64         
 16  Hail       755327 no

## 4. Creating tree of weather station locations

In [10]:
# Build KD-tree from unique weather station locations
unique_stations = weather_df[["LATITUDE", "LONGITUDE"]].drop_duplicates().reset_index(drop=True)
tree = cKDTree(unique_stations.values)

def get_nearest_station(lats, lons):
    _, indices = tree.query(list(zip(lats, lons)), k=1)
    return unique_stations.iloc[indices]["LATITUDE"].values, unique_stations.iloc[indices]["LONGITUDE"].values

# Find nearest station for each unique origin airport
unique_origins = flights_df[["origin_code", "origin_lat", "origin_lon"]].drop_duplicates().dropna().copy()
unique_origins["orig_wx_lat"], unique_origins["orig_wx_lon"] = get_nearest_station(unique_origins["origin_lat"], unique_origins["origin_lon"])

# Find nearest station for each unique destination airport
unique_dests = flights_df[["destination_code", "dest_lat", "dest_lon"]].drop_duplicates().dropna().copy()
unique_dests["dest_wx_lat"], unique_dests["dest_wx_lon"] = get_nearest_station(unique_dests["dest_lat"], unique_dests["dest_lon"])

print(f"Unique origin airports matched: {len(unique_origins)}")
print(f"Unique destination airports matched: {len(unique_dests)}")
print(unique_origins.head())
print(unique_dests.head())

Unique origin airports matched: 94
Unique destination airports matched: 172
       origin_code  origin_lat  origin_lon  orig_wx_lat  orig_wx_lon
0              EWR   40.689400  -74.170545     40.68275    -74.16927
75733          SFO   37.619806 -122.374821     37.61962   -122.36562
157872         ORD   41.978600  -87.904800     41.96017    -87.93164
317484         LGA   40.777199  -73.872597     40.77945    -73.88027
381530         JFK   40.639447  -73.779317     40.63915    -73.76390
     destination_code   dest_lat    dest_lon  dest_wx_lat  dest_wx_lon
0                 DTW  42.213770  -83.353786     42.23113    -83.33121
1                 SLC  40.788860 -111.979866     40.77069   -111.96503
2                 MSP  44.880081  -93.221741     44.88523    -93.23133
5                 ATL  33.636700  -84.428101     33.62972    -84.44224
6582              AUS  30.197535  -97.662015     30.18311    -97.67989


## 5. Combining Data using tree

In [12]:
df = flights_df.copy()
df["date"] = pd.to_datetime(df["date"])

# Merge nearest station coords onto flights
df = df.merge(unique_origins[["origin_code", "orig_wx_lat", "orig_wx_lon"]], on="origin_code", how="left")
df = df.merge(unique_dests[["destination_code", "dest_wx_lat", "dest_wx_lon"]], on="destination_code", how="left")

# Join origin weather by date + nearest station coords
origin_weather = weather_df.rename(columns={c: "origin_" + c for c in weather_df.columns if c not in ["DATE", "LATITUDE", "LONGITUDE"]})
origin_weather = origin_weather.rename(columns={"LATITUDE": "orig_wx_lat", "LONGITUDE": "orig_wx_lon"})
df = df.merge(origin_weather, left_on=["date", "orig_wx_lat", "orig_wx_lon"], right_on=["DATE", "orig_wx_lat", "orig_wx_lon"], how="left")
df = df.drop(columns=["DATE", "orig_wx_lat", "orig_wx_lon"])

# Join destination weather by date + nearest station coords
dest_weather = weather_df.rename(columns={c: "dest_" + c for c in weather_df.columns if c not in ["DATE", "LATITUDE", "LONGITUDE"]})
dest_weather = dest_weather.rename(columns={"LATITUDE": "dest_wx_lat", "LONGITUDE": "dest_wx_lon"})
df = df.merge(dest_weather, left_on=["date", "dest_wx_lat", "dest_wx_lon"], right_on=["DATE", "dest_wx_lat", "dest_wx_lon"], how="left")
df = df.drop(columns=["DATE", "dest_wx_lat", "dest_wx_lon"])

print(df.shape)
df.head()

(3966862, 48)


,Carrier Code,date,destination_code,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Taxi-Out time (Minutes),origin_code,month,day_of_week,...,dest_PRCP,dest_SNDP,dest_SLP,dest_ELEVATION,dest_Fog,dest_Rain,dest_Snow,dest_Hail,dest_Thunder,dest_Tornado
0,DL,2024-01-01,DTW,110.0,92.0,12.0,12.0,EWR,1.0,0.0,...,0.02,NaN,1020.3,191.9,0.0,1.0,1.0,0.0,0.0,0.0
1,DL,2024-01-01,SLC,325.0,291.0,-5.0,23.0,EWR,1.0,0.0,...,0.00,NaN,1023.3,1288.4,0.0,0.0,0.0,0.0,0.0,0.0
2,DL,2024-01-01,MSP,184.0,152.0,-10.0,13.0,EWR,1.0,0.0,...,0.02,1.2,1026.8,254.5,0.0,0.0,0.0,0.0,0.0,0.0
3,DL,2024-01-01,MSP,194.0,161.0,4.0,20.0,EWR,1.0,0.0,...,0.02,1.2,1026.8,254.5,0.0,0.0,0.0,0.0,0.0,0.0
4,DL,2024-01-01,SLC,330.0,278.0,18.0,17.0,EWR,1.0,0.0,...,0.00,NaN,1023.3,1288.4,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
df.to_csv("combined.csv", index=False)